# Bayesian Real Estate Intelligence
## Probabilistic Models for Multi-Portal Market Analysis

This notebook walks through the complete Bayesian modeling pipeline for analyzing
real estate listings from 4 Spanish portals. We build three complementary models:

1. **Hierarchical Pricing** — partial pooling across portals to share information
2. **Gaussian Process Spatial** — non-parametric price surfaces with calibrated uncertainty
3. **Anomaly Detection** — Bayesian mixture model to flag suspicious listings

Each model is validated with posterior predictive checks and convergence diagnostics.

### Why Bayesian?

Frequentist approaches give point estimates. Bayesian inference gives **full posterior distributions**,
meaning we get calibrated uncertainty for every prediction. For real estate pricing, knowing that a
property is worth "between 180k and 220k with 94% probability" is far more useful than a single
number with a p-value.

In [ ]:
import os, sys, warnings
os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="pymc")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 100})

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

---
## 1. Data Exploration

We generate synthetic listings that mimic the structure of our real scraped data:
4 portals (habitaclia, fotocasa, milanuncios, idealista), 8 zones in the Tarragona
coast, with realistic price distributions and 5% injected anomalies.

In [ ]:
from data import generate_synthetic, preprocess

df_raw = generate_synthetic(n_listings=800, n_anomalies=40)
df = preprocess(df_raw)

print(f"Listings: {len(df)}")
print(f"Portals:  {df['portal'].nunique()} — {', '.join(df['portal'].unique())}")
print(f"Zones:    {df['zone'].nunique()}")
print(f"Anomalies: {df['is_anomaly'].sum()} ({df['is_anomaly'].mean():.1%})")
print(f"\nPrice range: {df['price'].min():,.0f} – {df['price'].max():,.0f} EUR")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Price distribution by portal
for portal in df["portal"].unique():
    mask = df["portal"] == portal
    axes[0].hist(df.loc[mask, "price"], bins=30, alpha=0.5, label=portal, density=True)
axes[0].set_xlabel("Price (EUR)")
axes[0].set_title("Price Distribution by Portal")
axes[0].legend(fontsize=8)

# Price vs size
scatter = axes[1].scatter(df["size_m2"], df["price"], c=df["is_anomaly"].astype(int),
                          cmap="RdYlGn_r", s=15, alpha=0.6)
axes[1].set_xlabel("Size (m²)")
axes[1].set_ylabel("Price (EUR)")
axes[1].set_title("Price vs Size (red = anomaly)")

# Spatial distribution
axes[2].scatter(df["lon"], df["lat"], c=df["log_price"], cmap="RdYlGn_r", s=10, alpha=0.6)
axes[2].set_xlabel("Longitude")
axes[2].set_ylabel("Latitude")
axes[2].set_title("Geographic Distribution (color = log price)")

fig.tight_layout()
plt.show()

### Key observations

- Portals have **slightly different price distributions** — some portals skew higher (fotocasa, idealista)
  while others skew lower (milanuncios). This is the portal bias our hierarchical model will capture.
- The **price-size relationship is roughly linear in log space**, but with notable outliers — our anomaly
  detection model should flag these.
- There's clear **spatial structure** in prices (coastal areas are more expensive), which the GP model
  will learn automatically.

---
## 2. Hierarchical Pricing Model

The core insight is **partial pooling**: instead of fitting 4 independent models (complete pooling = no
portal effects; no pooling = independent per portal), we let portal-level parameters be drawn from a
shared group distribution:

$$\alpha_j \sim \text{Normal}(\mu_\alpha, \sigma_\alpha) \quad \text{for } j = 1, \ldots, J \text{ portals}$$

Portals with few listings get **shrunk toward the group mean** (borrowing strength),
while portals with many listings retain their individual estimates.

In [ ]:
from models import HierarchicalPricingModel

hier = HierarchicalPricingModel(df)
hier.build()

# Visualize the model structure
pm.model_to_graphviz(hier.model)

In [ ]:
# Sample with NUTS (No-U-Turn Sampler) — the gold standard for MCMC
hier.sample_nuts(draws=1000, tune=500, chains=2, cores=1)
print(f"NUTS completed in {hier.nuts_time:.1f}s")

# Also fit ADVI for tractability comparison
hier.sample_advi(n_iterations=20000)
print(f"ADVI completed in {hier.advi_time:.1f}s")
print(f"Speedup: {hier.nuts_time / hier.advi_time:.1f}x")

In [ ]:
# Convergence diagnostics
summary = az.summary(hier.trace, var_names=["mu_alpha", "mu_beta_size", "sigma", "alpha"])
print("Key diagnostics:")
print(f"  R-hat max:    {summary['r_hat'].max():.4f} (want < 1.05)")
print(f"  ESS bulk min: {summary['ess_bulk'].min():.0f} (want > 100)")
summary

### Shrinkage Effect

The defining feature of hierarchical models: portal estimates are "pulled" toward the group mean.
The degree of shrinkage depends on the amount of data per portal — portals with fewer listings
are shrunk more.

In [ ]:
from plots import plot_shrinkage

shrinkage = hier.shrinkage_summary()
plot_shrinkage(shrinkage)
plt.show()

print("\nShrinkage summary:")
print(shrinkage.to_string(index=False))

---
## 3. Posterior Predictive Checks (PPC)

A fitted model means nothing if it can't reproduce the observed data. PPC generates new
"fake" datasets from the posterior and compares them to the real data:

- If the simulated data looks like the real data → the model captures the data-generating process
- If they diverge → the model is misspecified (wrong functional form, missing features, etc.)

This is the Bayesian equivalent of residual analysis, but more powerful because it accounts
for parameter uncertainty.

In [ ]:
from plots import plot_ppc

hier.posterior_predictive_check()
ppc_metrics = hier.ppc_summary()

print("PPC Calibration:")
print(f"  94% interval coverage: {ppc_metrics['coverage_94pct']:.1%}")
print(f"  Mean absolute residual: {ppc_metrics['mean_abs_residual']:.4f}")
print(f"  Predicted std: {ppc_metrics['pp_mean_std']:.4f} vs Observed std: {ppc_metrics['observed_std']:.4f}")

plot_ppc(hier.trace, model_name="Hierarchical")
plt.show()

**Interpretation**: If the blue (predicted) and black (observed) distributions overlap well,
the model is well-calibrated. Coverage near 94% means our credible intervals are neither
overconfident (too narrow) nor too conservative (too wide).

---
## 4. Gaussian Process Spatial Model

Instead of manually encoding location features (distance to coast, city center, etc.),
we use a **Gaussian Process** to learn the spatial price function directly from coordinates.

The GP uses a **Matern-5/2 kernel**, which assumes the price surface is twice-differentiable
(smooth but not infinitely smooth — appropriate for real spatial data):

$$k(x, x') = \eta^2 \left(1 + \frac{\sqrt{5} |x-x'|}{\ell} + \frac{5|x-x'|^2}{3\ell^2}\right) \exp\left(-\frac{\sqrt{5}|x-x'|}{\ell}\right)$$

**Tractability note**: GPs scale as $O(n^3)$ in the number of data points, making them the
computational bottleneck. We subsample to 300 points.

In [ ]:
from models import SpatialGPModel
from plots import plot_spatial_surface

spatial = SpatialGPModel(df, max_points=200)
spatial.build()

spatial.sample_nuts(draws=500, chains=2)
print(f"GP sampling: {spatial.nuts_time:.1f}s")
print(spatial.summary())

In [ ]:
# Generate price surface prediction
grid = spatial.predict_grid(n_grid=25)
plot_spatial_surface(grid, df)
plt.show()

**Key result**: The right panel shows **prediction uncertainty** — it grows in areas with few
data points. This is a natural property of GPs and extremely valuable: we know *where* our
model is confident and where it isn't, without any ad-hoc decisions.

---
## 5. Anomaly Detection

We model the residual price distribution as a **two-component Bayesian mixture**:
- Component 1 (normal market): tight distribution around expected price
- Component 2 (anomaly): wide distribution capturing outliers

The posterior probability $P(z_i = \text{anomaly} \mid \text{data})$ gives a principled,
**calibrated** anomaly score — unlike threshold-based methods, it accounts for parameter uncertainty.

In [ ]:
from models import AnomalyMixtureModel
from plots import plot_anomaly_scores

anomaly = AnomalyMixtureModel(df)
anomaly.build()
anomaly.sample_nuts(draws=1000, tune=500, chains=2, cores=1)
print(f"Anomaly model: {anomaly.nuts_time:.1f}s")

scores = anomaly.anomaly_scores()
n_flagged = scores['is_flagged'].sum()
print(f"\nFlagged: {n_flagged}/{len(scores)} ({100*n_flagged/len(scores):.1f}%)")

# Detection performance against known ground truth
perf = anomaly.detection_performance()
print(f"Precision: {perf['precision']:.3f}")
print(f"Recall:    {perf['recall']:.3f}")
print(f"F1:        {perf['f1_score']:.3f}")

In [ ]:
plot_anomaly_scores(scores)
plt.show()

print("\nTop 10 most anomalous listings:")
scores.head(10)

---
## 6. Out-of-Sample Validation

All the above shows the model fits the training data well — but does it **generalize**?
We split the data 80/20 (stratified by portal), train on the 80%, and evaluate predictions
on the held-out 20%.

We report:
- **RMSE / MAE**: point prediction accuracy
- **MAPE**: percentage error in EUR (interpretable)
- **94% CI coverage**: are our uncertainty intervals well-calibrated on unseen data?

In [ ]:
from validation import run_validation, plot_predictions

val = run_validation(df, draws=1000, tune=500, chains=2, quick=False)

plot_predictions(val)
plt.show()

**Interpretation**:
- Points clustered along the diagonal → good predictions
- Coverage near 94% → well-calibrated uncertainty (not overconfident)
- Narrow intervals with good coverage → the model is both **sharp** and **calibrated**

---
## 7. Tractability Comparison

One of the practical challenges of Bayesian inference is computational cost.
We compare NUTS (exact MCMC) vs ADVI (fast variational approximation):

| Method | Pros | Cons |
|--------|------|------|
| NUTS   | Exact posterior, convergence guarantees | Slow, especially for GPs |
| ADVI   | Fast, scalable | Approximation, underestimates uncertainty |

**ESS/second** (effective samples per second) measures sampling efficiency — higher is better.

In [ ]:
from diagnostics import convergence_report, tractability_report

models = {"hierarchical": hier, "spatial": spatial, "anomaly": anomaly}

print("=" * 50)
print("CONVERGENCE")
print("=" * 50)
for name, m in models.items():
    report = convergence_report(m.trace, name)
    status = "PASS" if report["converged"].iloc[0] else "FAIL"
    print(f"  [{status}] {name} — R-hat: {report['r_hat_max'].iloc[0]}, "
          f"ESS: {report['ess_bulk_min'].iloc[0]}, Div: {report['divergences'].iloc[0]}")

print(f"\n{'=' * 50}")
print("TRACTABILITY")
print("=" * 50)
print(tractability_report(models).to_string(index=False))

---
## 8. Conclusions

### What we demonstrated

1. **Hierarchical modeling** enables information sharing across data sources (portals) while
   respecting their individual characteristics. The shrinkage effect is the key mechanism.

2. **Gaussian Processes** learn spatial structure without manual feature engineering, and provide
   calibrated uncertainty that grows in data-sparse regions.

3. **Bayesian anomaly detection** gives probabilistic scores instead of binary labels, enabling
   principled decision-making about flagging thresholds.

4. **Posterior predictive checks** validate that models can reproduce observed data — a fundamental
   step that's often skipped in applied ML.

5. **Out-of-sample validation** confirms the model generalizes beyond training data, with
   well-calibrated uncertainty intervals.

### Relevance to materials science

The same probabilistic framework applies directly to materials research:
- **Hierarchical models** → pooling across experimental batches or measurement devices
- **GPs** → modeling material property surfaces over composition/processing parameters
- **Anomaly detection** → identifying defective specimens or measurement errors
- **Uncertainty quantification** → critical for any scientific conclusion

### Technical stack

- **PyMC 5** + **ArviZ** for probabilistic programming and diagnostics
- **nutpie** (Rust-based) sampler for faster NUTS when available
- **NUTS vs ADVI** comparison for tractability analysis
- Convergence diagnostics: R-hat, ESS (bulk/tail), divergences
- Model comparison: LOO-CV, WAIC